Приветствую!  
Вашему вниманию полный pipeline Агента поддержки Lamoda Seller Academy.  
Код требует время на первый запуск в сессии, так как ему нужно скачать все зависимости.  
Идея в том, чтобы парсить базу знаний с сайта не при каждом запросе, а время от времени. При таком подходе мы сможем "кэшировать" все необходимые нам данные.  
Если проще, раз в неделю (или по надобности) будем парсить БЗ. У нас будет готовая директория со всеми статьями. Дальше мы обрабатываем эти статьи, у нас будут 3 варианта чанков (так нужно проекту) и их просчитаные эмбендинги и ключевые слова.  
Это дает нам скорость работы агента, все нужные данные будут готовы заранее и агент будет отвечать моментально.  
Для примера взял за основу LLM Gemini Flash Lite, именно он будет мозгом в этом проекте. Он сможет обратиться к нашей БЗ до 5 раз (если посчитает нужным).

In [ ]:
!pip install -q requests beautifulsoup4 markdownify
!pip install pymorphy3 nltk
!pip install rank_bm25
!pip install sentence-transformers
!pip install google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 73.0 MB/s eta 0:00:00


In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import os
import time
import re
import json
from markdownify import markdownify as mdify
from pathlib import Path
import nltk
import pymorphy3
from nltk.corpus import stopwords
from rank_bm25 import BM25Okapi
import torch
from sentence_transformers import SentenceTransformer, util
from google import genai
from google.genai import types
import shutil
from google.colab import userdata

In [ ]:
# Удаляем старую директорию во избежании ошибок
articles_dir = Path("./articles")
if articles_dir.exists():
    shutil.rmtree(articles_dir)
    print("Старая папка 'articles' удалена.")

# Удаляем старую директорию во избежании ошибок
chunks_dir = Path("./chunks")
if chunks_dir.exists():
    shutil.rmtree(chunks_dir)
    print("Старая папка 'chunks' удалена.")

Меню может иметь произвольную глубину вложенности. Рекурсивный обход гарантирует, что мы не пропустим ни один раздел, даже если структура изменится.

In [ ]:
BASE_URL = "https://academy.lamoda.ru"
START_URL = BASE_URL + "/articles/start-working/"

def get_all_section_urls() -> list:
    resp = requests.get(START_URL, timeout=10, headers={'User-Agent': 'Mozilla/5.0'})
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')
    sidebar = soup.find('div', id='sidebar')
    if not sidebar:
        raise Exception("Боковое меню не найдено")
    urls = set()
    def traverse(ul):
        for li in ul.find_all('li', recursive=False):
            a = li.find('a', href=True)
            if a:
                full = urljoin(BASE_URL, a['href'])
                if full.startswith(BASE_URL) and '/articles/' in full:
                    urls.add(full)
            inner = li.find('ul')
            if inner:
                traverse(inner)
    top_ul = sidebar.find('ul', class_='sidebar-menu-list')
    if top_ul:
        traverse(top_ul)
    return sorted(urls)

section_urls = get_all_section_urls()
print(f"Найдено {len(section_urls)} разделов")

Найдено 82 разделов


Страницы разделов используют два типа блоков со статьями, собираем оба.
Некоторые страницы могут иметь кнопку "Показать еще", учитываем это.
Lamoda может временно блокировать частые запросы. При получении 503 делаем повторный запрос после паузы (экспоненциальная задержка). Это повышает надёжность сбора.

In [ ]:
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

def get_article_urls_from_section(section_url: str) -> list:
    urls = []
    current_url = section_url
    page = 1

    while True:
        try:
            resp = requests.get(current_url, headers=HEADERS, timeout=15)
            if resp.status_code == 503:
                print(f"Ошибка 503 для {current_url}, повторяю запрос")
                time.sleep(10)
                resp = requests.get(current_url, headers=HEADERS, timeout=15)
            resp.raise_for_status()
            soup = BeautifulSoup(resp.text, 'html.parser')

            # Ссылки на статьи могут быть в двух разных разделах
            found = 0
            for a in soup.find_all('a', class_=['subsections-article-item__detail-link', 'current-article-item__detail-link']):
                href = a.get('href')
                if href:
                    full = urljoin(BASE_URL, href)
                    if '/articles/' in full:
                        urls.append(full)
                        found += 1
            print(f"    Страница {page}: найдено {found} статей")

            # Проверяем наличие кнопки "Показать еще"
            load_more = soup.find('div', class_='load-more')
            if load_more and load_more.get('data-url'):
                next_relative = load_more['data-url']
                current_url = urljoin(BASE_URL, next_relative)
                page += 1
                time.sleep(0.5)
            else:
                break
        except Exception as e:
            print(f"Ошибка при обработке {current_url}: {e}")
            break

    return urls

In [ ]:
all_article_urls = set()
for idx, sec in enumerate(section_urls):
    print(f"[{idx+1}/{len(section_urls)}] {sec}")
    arts = get_article_urls_from_section(sec)
    print(f"Статей: {len(arts)}")
    for a in arts:
        all_article_urls.add(a)
    time.sleep(0.5)

all_article_urls = sorted(all_article_urls)
print(f"\nВсего уникальных статей: {len(all_article_urls)}")

[1/82] https://academy.lamoda.ru/articles/aktsii/
    Страница 1: найдено 10 статей
Статей: 10
[2/82] https://academy.lamoda.ru/articles/api/
    Страница 1: найдено 0 статей
Статей: 0
[3/82] https://academy.lamoda.ru/articles/api/1-vvedenie/
    Страница 1: найдено 4 статей
Статей: 4
[4/82] https://academy.lamoda.ru/articles/api/10-fbo/
    Страница 1: найдено 4 статей
Статей: 4
[5/82] https://academy.lamoda.ru/articles/api/11-notifikatsii/
    Страница 1: найдено 4 статей
Статей: 4
[6/82] https://academy.lamoda.ru/articles/api/12-dostavka/
    Страница 1: найдено 3 статей
Статей: 3
[7/82] https://academy.lamoda.ru/articles/api/13-vozvraty/
    Страница 1: найдено 2 статей
Статей: 2
[8/82] https://academy.lamoda.ru/articles/api/14-voprosy-o-tovarakh/
    Страница 1: найдено 1 статей
Статей: 1
[9/82] https://academy.lamoda.ru/articles/api/15-markirovka/
    Страница 1: найдено 3 статей
Статей: 3
[10/82] https://academy.lamoda.ru/articles/api/16-reshenie-problem/
    Страница 1: найдено

Собираем ТОЛЬОК статьи, убираем все лишнее. Так же на будущее, для ссылок на картинки, восстонавливем абсолютный путь.

In [ ]:
def fetch_article(url: str, retries=3) -> dict:
    for attempt in range(retries):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=15)
            if resp.status_code == 503:
                print(f"Ошибка 503 {url}, попытка {attempt+1}")
                time.sleep(5 * (attempt + 1))
                continue
            resp.raise_for_status()
            soup = BeautifulSoup(resp.text, 'html.parser')

            # Заголовок H1
            h1 = soup.find('h1')
            title = h1.get_text(strip=True) if h1 else "Без заголовка"

            # Дата обновления
            date_str = ""
            date_div = soup.find('div', class_='article-detail-date__value')
            if date_div:
                date_span = date_div.find('span', class_='body-s')
                if date_span:
                    date_str = date_span.get_text(strip=True)

            # Теги
            tags = []
            for tag_span in soup.find_all('span', class_='article-detail__tag'):
                a = tag_span.find('a')
                if a:
                    tags.append(a.get_text(strip=True))

            # Основной контент
            content_div = soup.find('div', class_='article-detail__text')
            if not content_div:
                # Запасной вариант
                content_div = soup.find('div', class_='article-detail-content')
            if not content_div:
                content_div = soup.find('body')

            # Очистка мусора
            for bad in content_div.find_all(['script', 'style', 'footer', 'nav']):
                bad.decompose()
            for cls in ['article-detail-share-flex', 'article-feedback', 'article-detail-bottom',
                        'article-detail-prev-next-container', 'article-detail-page-nav',
                        'article-detail-tags-date', 'component-main-header', 'breadcrumb-title-container',
                        'back-link-container']:
                for elem in content_div.find_all(class_=cls):
                    elem.decompose()
            # Удаляем верхний заголовок H1, если он дублируется внутри контента
            for h in content_div.find_all('h1'):
                h.decompose()

            # Создаем полную ссылку на картинки
            for img in content_div.find_all('img'):
                src = img.get('src')
                if src:
                    img['src'] = urljoin(url, src)

            html_content = str(content_div)
            md_text = mdify(html_content, heading_style="ATX", strip=['script', 'style'])
            # Убираем пустые строки
            md_text = re.sub(r'\n{3,}', '\n\n', md_text).strip()

            return {
                "url": url,
                "title": title,
                "last_update": date_str,
                "tags": tags,
                "content_md": md_text
            }
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(5)
                continue
            print(f"Ошибка {url}: {e}")
            return {}
    return {}

In [ ]:
OUTPUT_DIR = "articles"
os.makedirs(OUTPUT_DIR, exist_ok=True)

metadata_list = []

for idx, url in enumerate(all_article_urls):
    print(f"[{idx+1}/{len(all_article_urls)}] {url}")
    article = fetch_article(url)
    if not article:
        continue

    # Создаем имя файла используя slug
    slug = url.rstrip('/').split('/')[-1]
    if not slug or slug == 'articles':
        parts = url.rstrip('/').split('/')
        slug = parts[-2] if len(parts) >= 2 else 'index'
    filename = f"{slug}.md"
    filepath = os.path.join(OUTPUT_DIR, filename)

    # Формируем только нужный контент в md файлах
    md_content = f"# {article['title']}\n\n{article['content_md']}"

    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(md_content)

    metadata_list.append({
        "url": article['url'],
        "title": article['title'],
        "last_update": article['last_update'],
        "tags": article['tags'],
        "file": filepath
    })
    time.sleep(0.3)

print(f"\nСохранено {len(metadata_list)} статей в '{OUTPUT_DIR}'")

with open('articles_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata_list, f, ensure_ascii=False, indent=2)
print("articles_metadata.json создан")

[1/293] https://academy.lamoda.ru/articles/aktsii/aktsii-s-dopolnitelnoy-skidkoy-ot-lamoda/
[2/293] https://academy.lamoda.ru/articles/aktsii/aktsii-s-lyuboy-skidkoy/
[3/293] https://academy.lamoda.ru/articles/aktsii/instruktsiya-po-rabote-s-edinym-excel-shablonom-v-aktsiyakh/
[4/293] https://academy.lamoda.ru/articles/aktsii/kak-dobavit-tovary-v-aktsiyu-cherez-interfeys/
[5/293] https://academy.lamoda.ru/articles/aktsii/kak-udalit-tovary-iz-aktsii-cherez-interfeys/
[6/293] https://academy.lamoda.ru/articles/aktsii/pravila-izmeneniya-aktsionnoy-skidki-u-tovara-kotoryy-uzhe-uchastvuet-v-aktsii/
[7/293] https://academy.lamoda.ru/articles/aktsii/razdel-aktsii-osnovnaya-informatsiya/
[8/293] https://academy.lamoda.ru/articles/aktsii/targetirovannye-aktsii/
[9/293] https://academy.lamoda.ru/articles/aktsii/tovary-s-potentsialom-uluchsheniya-metrik-za-schet-uchastiya-v-aktsiyakh/
[10/293] https://academy.lamoda.ru/articles/aktsii/validatsiya-aktsionnykh-tsen/
[11/293] https://academy.lamoda.

Делаем из нашей базы знананий наборы чанков. Нам нужны как raw чанки для подачи в LLM, так и очищенные от ссылок и спец символов, для работы ретриверов. Делаем чанкирование по строкам и стараемся каждый чанк начинать с начала предложения. По строкам делаем, потому что так намного удобнее будет готовить качественные чанки для ретриверов.  

**raw** – оригинальный Markdown с таблицами, ссылками, переносами. Подаётся в LLM.  

**clean** – удалены ссылки, картинки, Markdown-разметка, пайпы заменены на ;. Используется для dense-энкодера.  

**clean-sparse** – дополнительно лемматизирован, убраны стоп-слова. Используется для BM25.

In [ ]:
def clean_for_search(text: str) -> str:
    # Превращаем пайпы таблиц в точки с запятой
    text = text.replace('|', ';')

    # Вырезаем любые остатки картинок, битых ссылок и путей к файлам (.png, .jpg)
    text = re.sub(r'!\[[^\]]*\](?:\([^\)]*\))?', '', text) # Полные и битые теги ![alt](url)
    text = re.sub(r'\[(.*?)\]\([^\)]*\)', r'\1', text)    # Ссылки [текст](url) оставляем только "текст"
    text = re.sub(r'(?i)\b(?:https?|tps)://\S+', '', text) # Любые URL/битые урлы
    text = re.sub(r'(?i)\b\S*?\.(?:png|jpg|jpeg|gif|webp)\b\S*', '', text) # Файлы картинок в тексте
    text = re.sub(r'(?i)"[^"]+\.(?:png|jpg|jpeg|gif|webp)"', '', text)     # Картинки в кавычках

    # Стираем маркеры Markdown
    text = re.sub(r'[#*_~`•\-]', ' ', text)
    text = text.replace('\xa0', ' ')

    # Схлопываем множественные разделители таблиц подряд в один
    text = re.sub(r';[\s;]*;', ';', text)

    # Превращаем всё в одну чистую строку без лишних пробелов
    text = re.sub(r'\s+', ' ', text).strip()
    return text.strip('; ') # Убираем ';' на краях, если остались

In [ ]:
def chunk_by_lines(text: str, max_lines=30, overlap_lines=4) -> list:
    lines = text.splitlines()
    total_lines = len(lines)
    chunks = []

    start = 0
    while start < total_lines:
        end = min(start + max_lines, total_lines)

        # Собираем строки текущего чанка
        chunk_text = "\n".join(lines[start:end]).strip()
        if chunk_text:
            chunks.append(chunk_text)

        if end == total_lines:
            break

        # Определяем дефолтную точку следующего старта с учетом оверлапа
        target_start = end - overlap_lines

        # Сканируем зону перекрытия в поисках начала логического предложения
        found_good_start = False
        for i in range(target_start, end):
            line = lines[i].strip()
            # Отрезаем мусор начала строки (цифры списков, маркеры, пробелы, пайпы), чтобы увидеть букву
            clean_start = re.sub(r'^[\s*+\-•|#\d.)\];]+', '', line).strip()

            # Если строка начинается с заглавной буквы это то что нам нада
            if clean_start and clean_start[0].isupper():
                start = i
                found_good_start = True
                break

        # Если заглавную букву не нашли, просто берем стандартный сдвиг
        if not found_good_start:
            start = target_start

    return chunks

In [ ]:
def process_articles(metadata_path: str, out_dir: str = "./chunks"):
    base = Path(out_dir)
    (base / 'raw').mkdir(parents=True, exist_ok=True)
    (base / 'clean').mkdir(parents=True, exist_ok=True)

    articles = json.loads(Path(metadata_path).read_text(encoding='utf-8'))
    meta, g_id = [], 0

    for art in articles:
        f_path = Path(art['file'])
        if not f_path.exists(): continue

        # Читаем оригинальный файл как есть
        source_text = f_path.read_text(encoding='utf-8')

        # Нарезаем строго по строкам
        chunks = chunk_by_lines(source_text, max_lines=30, overlap_lines=4)

        for i, r_content in enumerate(chunks):
            # Пропускаем совсем пустые куски
            if len(r_content.strip()) < 15: continue

            c_id = f"chunk_{g_id:05d}"

            # 1. RAW: Абсолютно нетронутый текст (таблицы и переносы строк сохранены)
            (base / f"raw/{c_id}.md").write_text(r_content, encoding='utf-8')

            # 2. CLEAN: Жестко вычищенный текст, пайпы заменены на ';'
            c_content = clean_for_search(r_content)
            (base / f"clean/{c_id}.txt").write_text(c_content, encoding='utf-8')

            # 3. Метаданные
            meta.append({
                **art,
                'chunk_id': c_id,
                'chunk_index': i,
                'raw_file': str(base / f"raw/{c_id}.md"),
                'clean_file': str(base / f"clean/{c_id}.txt")
            })
            g_id += 1

    (base / "chunks_metadata.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f"Готово! Создано {g_id} чанков.")

In [ ]:
# Запуск
process_articles('articles_metadata.json')

Готово! Создано 1118 чанков.


Далее делаем еще один набор чанков. Следующий набор будет исключительно для работы sparse ретривера, форматируем все чанки в формат, с которым он должен работать.

In [ ]:
# Скачиваем стоп-слова для русского языка
nltk.download('stopwords', quiet=True)

True

In [ ]:
def prepare_sparse_processor():
    morph = pymorphy3.MorphAnalyzer()
    stop_words = set(stopwords.words('russian'))

    return morph, stop_words

morph, stop_words = prepare_sparse_processor()

In [ ]:
def lemmatize_text(text: str, morph, stop_words) -> str:
    # Оставляем только буквы и цифры
    clean_words = re.findall(r'\b[a-zа-яё0-9-]+\b', text.lower())

    lemmatized = []
    for word in clean_words:
        if word not in stop_words and not word.isdigit():
            # Приводим слово к нормальной форме
            parsed = morph.parse(word)[0]
            lemmatized.append(parsed.normal_form)

    return " ".join(lemmatized)

In [ ]:
def process_sparse_directory(out_dir: str = "./chunks"):
    base = Path(out_dir)
    clean_dir = base / 'clean'
    sparse_dir = base / 'clean-sparse'
    sparse_dir.mkdir(parents=True, exist_ok=True)

    if not clean_dir.exists():
        print("Ошибка: Папка clean не найдена! Сначала запусти прошлый чанкер.")
        return

    morph, stop_words = prepare_sparse_processor()

    # Загружаем существующую мету, чтобы дополнить её путями к sparse-файлам
    meta_path = base / "chunks_metadata.json"
    if meta_path.exists():
        metadata = json.loads(meta_path.read_text(encoding='utf-8'))
    else:
        metadata = []

    meta_map = {item['chunk_id']: item for item in metadata}

    print("Запуск лемматизации для Sparse индекса...")
    count = 0

    for file_path in clean_dir.glob("*.txt"):
        chunk_id = file_path.stem
        text = file_path.read_text(encoding='utf-8')

        # Лемматизируем
        sparse_content = lemmatize_text(text, morph, stop_words)

        # Сохраняем в clean-sparse
        out_file = sparse_dir / f"{chunk_id}.txt"
        out_file.write_text(sparse_content, encoding='utf-8')

        # Обновляем метаданные, если этот чанк там есть
        if chunk_id in meta_map:
            meta_map[chunk_id]['sparse_file'] = str(out_file)

        count += 1

    # Перезаписываем обновленную мету
    if metadata:
        meta_path.write_text(json.dumps(list(meta_map.values()), ensure_ascii=False, indent=2), encoding='utf-8')

    print(f"Успешно обработано и сохранено {count} файлов в директорию 'clean-sparse'.")

In [ ]:
# Запуск этапа
process_sparse_directory()

Запуск лемматизации для Sparse индекса...
Успешно обработано и сохранено 1118 файлов в директорию 'clean-sparse'.


Собираем классы ретриверов. Sparse и Dense каждый может найти до 20 вариантов. Гибрид в конце выберит топ 10 основываясь на результатх каждого.

Для Sparse берем модель BM25. Модель очень хорошо себя зарекомендовала и в тандеме с хорошей и правильной обработкой чанков, дает очень хороший результат.

In [ ]:
class SparseRetriever:
    def __init__(self, chunks_dir: str = "./chunks"):
        self.base_dir = Path(chunks_dir)
        self.sparse_dir = self.base_dir / 'clean-sparse'

        # Загружаем метадату
        with open(self.base_dir / "chunks_metadata.json", "r", encoding="utf-8") as f:
            self.metadata = json.load(f)

        self.meta_map = {item['chunk_id']: item for item in self.metadata}
        self.chunk_ids = []
        self.corpus = []

        self._build_index()

    def _build_index(self):
        sparse_files = sorted(list(self.sparse_dir.glob("*.txt")))
        for file_path in sparse_files:
            chunk_id = file_path.stem
            text = file_path.read_text(encoding='utf-8')
            tokens = text.split()
            if tokens:
                self.chunk_ids.append(chunk_id)
                self.corpus.append(tokens)

        self.bm25 = BM25Okapi(self.corpus)
        print(f"[Sparse] Индекс собран. Готово к поиску.")

    def search(self, raw_query: str, morph, stop_words, top_k: int = 5, min_score: float = 1.0) -> list:
        from __main__ import lemmatize_text

        cleaned_query = lemmatize_text(raw_query, morph, stop_words)
        query_tokens = cleaned_query.split()

        if not query_tokens:
            return []

        doc_scores = self.bm25.get_scores(query_tokens)
        results = []

        top_indices = sorted(range(len(doc_scores)), key=lambda i: doc_scores[i], reverse=True)[:top_k]

        for idx in top_indices:
            score = doc_scores[idx]
            if score <= min_score:
                continue

            c_id = self.chunk_ids[idx]
            chunk_meta = self.meta_map[c_id]

            # Вытаскиваем мета информацию из json
            results.append({
                "score": round(float(score), 2),
                "title": chunk_meta.get("title", "Без названия"),
                "url": chunk_meta.get("url", "https://academy.lamoda.ru/"),
                "raw_file": chunk_meta["raw_file"],
                "last_update": chunk_meta.get("last_update", "(Дата неизвестна)")
            })

        return results

Для Dense берем rubert-tiny2. Отличная очень легкая модель, которая со своей работой справляется. Обучена на русском языке + знает базовые англ слова. Работает очень быстро, спокойно справится даже ЦП.  
(В будущем можно будет подобрать модель поинтереснее, если будет больше машинных ресурсов).

In [ ]:
class DenseRetriever:
    def __init__(self, chunks_dir: str = "./chunks", model_name: str = "cointegrated/rubert-tiny2"):
        self.base_dir = Path(chunks_dir)
        self.clean_dir = self.base_dir / 'clean'

        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"[Dense] Используем устройство: {self.device.upper()}")

        self.model = SentenceTransformer(model_name, device=self.device)

        # Загружаем метадату
        with open(self.base_dir / "chunks_metadata.json", "r", encoding="utf-8") as f:
            self.metadata = json.load(f)
        self.meta_map = {item['chunk_id']: item for item in self.metadata}

        self.chunk_ids = []
        self.embeddings = None

        self._build_index()

    def _build_index(self):
        clean_files = sorted(list(self.clean_dir.glob("*.txt")))
        texts_to_embed = []

        for file_path in clean_files:
            chunk_id = file_path.stem
            text = file_path.read_text(encoding='utf-8')
            if text.strip():
                self.chunk_ids.append(chunk_id)
                texts_to_embed.append(text)

        if not texts_to_embed:
            print("[Dense] Ошибка: Папка clean пуста.")
            return

        print(f"[Dense] Кодируем {len(texts_to_embed)} чанков")

        # Распараллелим задачу
        self.embeddings = self.model.encode(
            texts_to_embed,
            convert_to_tensor=True,
            batch_size=8,
            show_progress_bar=True
        )
        print("[Dense] Индексация завершена.")

    def search(self, raw_query: str, top_k: int = 5, min_similarity: float = 0.45) -> list:
        if self.embeddings is None:
            return []

        query_embedding = self.model.encode(raw_query, convert_to_tensor=True)
        cos_scores = util.cos_sim(query_embedding, self.embeddings)[0]

        top_results = torch.topk(cos_scores, k=min(top_k, len(cos_scores)))

        scores = top_results.values.cpu().tolist()
        indices = top_results.indices.cpu().tolist()

        results = []
        for score, idx in zip(scores, indices):
            if score <= min_similarity:
                continue

            c_id = self.chunk_ids[idx]
            chunk_meta = self.meta_map[c_id]

            results.append({
                "score": round(float(score), 4),
                "title": chunk_meta.get("title", "Без названия"),
                "url": chunk_meta.get("url", "https://academy.lamoda.ru/"),
                "raw_file": chunk_meta["raw_file"],
                "last_update": chunk_meta.get("last_update", "(Дата неизвестна)")
            })

        return results

Гибрид будет учитывать даже такие варианта, если один из ретриверов будет полностью уверен в нужности чанка, а второй даже не добавит его в свой список, такой чанк имеет право попасть в итоговый список. Все благодаря правильному составлению порога относительно RRF формулы.

In [ ]:
class HybridRetriever:
    def __init__(self, sparse_retriever, dense_retriever):
        self.sparse = sparse_retriever
        self.dense = dense_retriever

    def search(self, raw_query: str, morph, stop_words, top_k: int = 10, k_rrf: int = 60, min_rrf_score: float = 0.016) -> list:

        # 1. Запрашиваем ТОП-20 у каждого ретривера
        sparse_res = self.sparse.search(raw_query, morph, stop_words, top_k=20)
        dense_res = self.dense.search(raw_query, top_k=20)

        # Если вообще ничего не нашли оба ретривера
        if not sparse_res and not dense_res:
            return []

        rrf_scores = {}
        doc_data = {} # Хранилище метаданных

        # 2. Считаем баллы для результатов Sparse (BM25)
        for rank, doc in enumerate(sparse_res, start=1):
            doc_id = doc["raw_file"]
            doc_data[doc_id] = doc
            # Формула RRF: 1 / (k + rank)
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (1.0 / (k_rrf + rank))

        # 3. Считаем баллы для результатов Dense (rubert-tiny2)
        for rank, doc in enumerate(dense_res, start=1):
            doc_id = doc["raw_file"]
            # Если документа не было в выдаче sparse, сохраняем его метадату
            if doc_id not in doc_data:
                doc_data[doc_id] = doc
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (1.0 / (k_rrf + rank))

        # 4. Сортируем документы по накопленному RRF-скору (от большего к меньшему)
        sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

        # Фильтруем: оставляем только те чанки, у которых скор равен или выше порога
        filtered_docs = []
        for doc_id, score in sorted_docs:
            if score >= min_rrf_score:
                filtered_docs.append((doc_id, score))
            else:
                # Так как список отсортирован, как только пошел скор ниже порога — дальше идти нет смысла
                break

        # 5. Формируем финальный список
        final_results = []
        for doc_id, score in filtered_docs[:top_k]:
            item = doc_data[doc_id].copy()
            item["rrf_score"] = round(score, 4)
            final_results.append(item)

        return final_results

In [ ]:
# 2. Создаем экземпляры базовых ретриверов
retriever = SparseRetriever()
dense_retriever = DenseRetriever()

# 3. Собираем их в единый гибрид, который ищет функция
hybrid_retriever = HybridRetriever(sparse_retriever=retriever, dense_retriever=dense_retriever)

print("Все поисковые движки успешно инициализированы и готовы к работе!")

[Sparse] Индекс собран. Готово к поиску.
[Dense] Используем устройство: CPU


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.40k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[Dense] Кодируем 1118 чанков


Batches:   0%|          | 0/140 [00:00<?, ?it/s]

[Dense] Индексация завершена.
Все поисковые движки успешно инициализированы и готовы к работе!


In [ ]:
def search_knowledge_base(query: str, top_k: int = 10):
    # Вызываем гибридный ретривер (до 10 чанков за раз)
    return hybrid_retriever.search(query, morph, stop_words, top_k=top_k)

In [ ]:
# Ключ от Gemini
GOOGLE_API_KEY = input("Введите ваш Gemini API Key: ")

# Инициализируем новый клиент по официальному гайду Google
client = genai.Client(api_key=GOOGLE_API_KEY.strip())

Формируйем простой ReAct. Даем модели возможность делать поиск по базе до тех пор, пока она этого хочет. Результаты прошлых запросов и мыслей модели добавляем в историю, тем самым формируем промпт в котором есть история прошлых запросов. Следим за дублекатами, чтобы не тратить лишние токены на чанки, которые уже есть в ответах от ретривера. По итогу у нас получается нарастающий промпт, который может прожить до 5 запросов. Все мысли LLM тоже выводим, ответ прозрачен.  
Есть защита от галлюцинаций, код не должен падать.

In [ ]:
import re
from pathlib import Path

def run_lamoda_rag(user_question: str, model_name: str = "gemini-flash-lite-latest"):
    config = types.GenerateContentConfig(
        temperature=0.3,
        top_p=0.95,
        top_k=40
    )

    # Промпт, который подается модели в самом начале.
    react_system_prompt = (
        "Ты — ИИ-агент, сверхопытный ассистент поддержки Академии селлеров Lamoda.\n"
        "Твоя задача — дать максимально точный, развернутый и подтвержденный документами ответ на вопрос пользователя, используя инструмент поиска.\n\n"

        "ИНСТРУМЕНТ ДЛЯ ПОИСКА:\n"
        "У тебя есть доступ к функции поиска по базе знаний. Ты вызываешь её СТРОГО в таком формате на отдельной строке:\n"
        "Action: search_knowledge_base(query='твой поисковый запрос')\n"
        "Этот инструмент возвращает релевантные фрагменты статей. Передавай в query только чистый текст запроса, без лишних символов.\n\n"

        "ПРАВИЛА ЦИКЛА ReAct (ОБЯЗАТЕЛЬНЫЙ ФОРМАТ):\n"
        "Ты обязан строго соблюдать пошаговую структуру и выдавать её построчно:\n"
        "Thought: твои рассуждения (что ты ищешь, почему, анализ предыдущих шагов).\n"
        "Action: search_knowledge_base(query='текст запроса')\n\n"
        "ОБЯЗАТЕЛЬНОЕ ПРАВИЛО: Как только ты написал строку 'Action:', ты должен СРАЗУ остановить генерацию и дождаться ответа системы (Observation). Не пиши Observation сам!\n\n"

        "ЖЕСТКИЙ ЗАПРЕТ НА ОБРЫВЫ ФОРМАТА:\n"
        "Каждое твое сообщение ОБЯЗАНО заканчиваться ЛИБО строкой с вызовом 'Action:', ЛИБО строкой 'Final Answer:'.\n"
        "Ни в коем случае не останавливай генерацию после блока 'Thought:'. Если ты подумал — ты обязан либо сделать поиск, либо выдать итог.\n\n"

        "МЕГА-ПРАВИЛО УПОРСТВА И МНОГОЗАДАЧНОСТИ:\n"
        "1. База знаний огромна, но статьи могут использовать разные формулировки. Один или два неточных ответа не значат, что информации нет. Ты должен проявить упорство и использовать до 5 поисков.\n"
        "2. Если пользователь задал сразу несколько разнородных вопросов, РАЗБЕЙ ИХ для себя на подзадачи. Делай поисковые запросы поочередно для каждой темы, пока не соберешь полную картину.\n"
        "3. Если поиск не дал ответа, ты ОБЯЗАН менять стратегию, двигаясь от узких терминов к самым широким:\n"
        "   - Шаг 1: Прямой узкий запрос.\n"
        "   - Шаг 2: Смежные темы и синонимы.\n"
        "   - Шаг 3: Широкие темы.\n\n"
        "Ты имеешь право сдаться и написать, что информации нет, ТОЛЬКО если ты честно выполнил минимум 3-4 РАЗЛИЧНЫХ по смыслу запроса и проверил все уровни абстракции.\n\n"

        "ФИНАЛЬНЫЙ ОТВЕТ:\n"
        "Как только ты собрал информацию по ВСЕМ пунктам вопроса или исчерпал лимит в 5 разных поисков, пиши:\n"
        "Thought: Я обработал все части вопроса и готов сформировать итог.\n"
        "Final Answer: [Твой подробный, структурированный по пунктам ответ на основе ТОЛЬКО найденных документов. Если данных нет вообще по всем пунктам после 5 поисков, пиши строго: 'К сожалению, у меня нет информации по этому вопросу, попробуйте переформулировать ваш вопрос.']\n"
)

    agent_history = f"{react_system_prompt}\n\nВопрос пользователя: {user_question}\n"

    used_sources = []
    seen_files = set()
    search_counter = 0
    max_searches = 5

    print(f"Вопрос пользователя: {user_question}\n")
    print("--- Запуск ReAct цикла агента ---")

    final_answer = "Ошибка: Агент не смог сформировать ответ."

    for iteration in range(1, max_searches + 2):
        response_text = client.models.generate_content(
            model=model_name,
            contents=agent_history,
            config=config
        ).text.strip()

        # Првоерка вызов Action
        action_match = re.search(r"(Action:\s*search_knowledge_base\(query=['\"](.*?)['\"]\))", response_text, re.IGNORECASE)

        if action_match:
            full_action_line = action_match.group(1)
            query = action_match.group(2)

            end_pos = response_text.find(full_action_line) + len(full_action_line)
            clean_response = response_text[:end_pos].strip()

            agent_history += f"\n{clean_response}\n"

            for line in clean_response.split('\n'):
                if line.startswith("Thought:") or line.startswith("Action:"):
                    print(line)

            search_counter += 1
            if search_counter > max_searches:
                print(f"\n[Агент превысил лимит в {max_searches} поисков. Принудительный останов]")
                observation_str = "Observation: Ошибка. Превышен лимит запросов (5). Сформируй Final Answer."
                agent_history += f"\n{observation_str}\n"
                continue

            print(f"  -> Вызов ретривера с запросом: '{query}' (Поиск {search_counter}/{max_searches})")
            res = hybrid_retriever.search(query, morph, stop_words, top_k=10)
            print(f"     Ретривер вернул фрагментов: {len(res)}")

            step_contexts = []
            for chunk in res:
                file_id = chunk["raw_file"]
                raw_path = Path(file_id)

                source_info = {
                    "title": chunk["title"],
                    "url": chunk["url"],
                    "date": chunk.get("last_update") or "(Дата неизвестна)"
                }
                if source_info not in used_sources:
                    used_sources.append(source_info)

                if file_id not in seen_files:
                    seen_files.add(file_id)
                    if raw_path.exists():
                        full_text = raw_path.read_text(encoding='utf-8')
                        step_contexts.append(f"Статья: {chunk['title']}\nТекст:\n{full_text}")
                else:
                    step_contexts.append(f"Статья: {chunk['title']} (Контент этой статьи уже предоставлен выше)")

            if step_contexts:
                observation_str = "Observation:\n" + "\n---\n".join(step_contexts)
            else:
                observation_str = "Observation: По твоему запросу в базе знаний ничего не найдено. Попробуй использовать другие, более общие ключевые слова."

            agent_history += f"\n{observation_str}\n"
            continue

        # Првоерка Final Answer и защита от шизы, если модель отклонилась от курса и не соблюла токен маркеры
        elif "Final Answer:" in response_text:
            final_answer = response_text.split("Final Answer:")[-1].strip()
            for line in response_text.split('\n'):
                if line.startswith("Thought:"):
                    print(line)
            break

        else:
            print("[Внимание: Модель нарушила формат ReAct. Корректировка]")
            agent_history += "\nObservation: Ошибка формата. Используй строго структуру 'Thought:' -> 'Action: search_knowledge_base(query=...)' или 'Final Answer:'.\n"

    print("\nОтвет готов!")
    print(final_answer)

    if used_sources:
        print("\nДля более точной информации, воспользуйтесь источниками:")
        for src in used_sources:
            print(f"  - [{src['title']}]({src['url']}) от {src['date']}")
    else:
        print("\nИсточники: При поиске ретривер не вернул ни одного совпадения.")

Здесь можно написать свой запрос агенту и посмотреть на результат =)

In [ ]:
run_lamoda_rag("За что можно получить штраф и как избежать наказания? Есть ли скидки сотрудникам?") # Сюда вводите ваш запрос

Вопрос пользователя: За что можно получить штраф и как избежать наказания? Есть ли скидки сотрудникам?

--- Запуск ReAct цикла агента ---
Thought: Мне нужно найти информацию о штрафах для селлеров на Lamoda, способах их избежать, а также уточнить, существуют ли скидки для сотрудников селлеров. Начну с поиска информации о штрафах и системе санкций.
Action: search_knowledge_base(query='штрафы для селлеров Lamoda причины и как избежать')
  -> Вызов ретривера с запросом: 'штрафы для селлеров Lamoda причины и как избежать' (Поиск 1/5)
     Ретривер вернул фрагментов: 5
Thought: Я собрал подробную информацию о штрафах для селлеров (FBS и FBO) и причинах их начисления. Информации о скидках для сотрудников селлеров в базе знаний не найдено, поэтому я сделаю финальный запрос по этой теме, чтобы убедиться в отсутствии данных, прежде чем формировать ответ.
Action: search_knowledge_base(query='скидки для сотрудников партнеров селлеров')
  -> Вызов ретривера с запросом: 'скидки для сотрудников парт

Улучшать этот проект можно долго и в разыне стороны, но вот то, что я бы хотел отметить:  
1. Нужно поработать с парсингом и придумать его логику. Не обязательно парсить всегда весь сайт (это было сделано для упрощения в научных целях). Мы можем ориентироваться на дату обновлений статей, благо они есть на сайте и отлично сохраняются в метаданные. Мы можем ориентироваться на дату последнего парсинга и парсить только новые статьи. Да, нам всеравно придется пересчитывать Sparse, но это не так долго. По итогу мы не будем качать уже имеющиеся статьи, не будем высчитывать Dense на уже имеющихся статьях.  
2. В будущем хотелось бы эмбеддер посерьезнее. Да rubert-tiny2 неплох для показательной работы, но если пускать в прод, нужно менять. В будущем у нас будет и железо лучше и в целом все векторы будем кэшировать, так что просчитать один раз на мощном эмбеддере будет не проблема. Зато это нам даст тот самы тонкий контекст, которого сейчас не хватает, для более качественного ответа.
3. Улучшения, которые зависят от LLM. Так как основная LLM будет другая, нужно это учитывать. Под новую LLM будем подгонять размер чанков, настройки декодерной стратегии, сам промпт. Здесь мы можем пробовать даже использовать картинки из статей, скачивать их или смотреть через ссылки. Это важно, потому что во многих статьях именно в картинках хранится нужная и важная информация. Если мы разрешаем нашей новой LLM выходить в мировую паутину, то при парсинге делаем все ссылки (на картинки и гиперссылки) абсолютными, это нам даст более богатую картину ответа. Количество возможных запросов от LLM к ретриверу тоже зависит от размера чанков и его контекстного окна.  
4. Еще как вариант работы с картинками можно сделать проще. Специальной моделью будем переводить весь текст из картинки в текст. У нас будет кэш всех картинок в виде текста. Это так же упростит работу нашего агента + увеличит время ответа + сэкономит наши ресурсы.    

По итогу хочу еще отметить на счет чанкирования. Мы делаем чанки именно по строкам. Это удобнее в нашем случае, потому что в статьях много таблиц, картинок, спец символов и вообще непоследовательного текста. Такое будет сложно делить и подготавливать по тем же абзацам не ломая смысл текста.  
Спасибо за уделенное время!